### Install Automatic1111 Web UI
- First, clone the official Web UI repository into Kaggle’s working directory.

In [ ]:
!git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui /kaggle/working/stable-diffusion-webui
%cd /kaggle/working/stable-diffusion-webui
!git switch dev
!git pull

### Download Stable Diffusion 1.5 Base Model
- Fetch the v1.5 checkpoint and a VAE (Variational Autoencoder) for better image quality.

In [ ]:
!mkdir -p /kaggle/temp/models
!wget -O models/Stable-diffusion/v1-5-pruned-emaonly.safetensors "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
!wget -O models/VAE/vae-ft-mse-840000-ema-pruned.safetensors "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors"

### Install Dependencies
- Uninstall any pre-installed Torch versions and install the exact ones needed for our GPU setup, along with Web UI requirements and ngrok.


In [ ]:
!pip uninstall -y torch torchvision torchaudio xformers peft accelerate diffusers transformers
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124
!pip install -r /kaggle/working/stable-diffusion-webui/requirements_versions.txt
!pip install -qq pyngrok

### Restart Session
- Restarting ensures Kaggle reloads the correct GPU libraries before we launch the Web UI (Important for GPU initialization).


### Configure ngrok
- Ngrok creates a secure URL so we can access the Web UI running in Kaggle from our browser

In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token (get it from ngrok.com)
ngrok.set_auth_token("Your_ngrok_token")

### Move into the directory
- As we restarted the session, we need to move to the web-ui folder to start the UI


In [ ]:
%cd /kaggle/working/stable-diffusion-webui

### Launch the Web UI
- Finally, we run the launch command to start Stable Diffusion Web UI and open it through ngrok

In [ ]:
import threading
import time
import subprocess
import requests # New import
from pyngrok import ngrok

def kill_existing_tunnels():
    """Kill all existing ngrok tunnels to free up slots."""
    try:
        ngrok.disconnect_all()
        print("All existing ngrok tunnels disconnected.")
    except Exception as e:
        print(f"Error disconnecting tunnels: {e}")

def run_webui():
    """Function to run the Automatic1111 WebUI."""
    # Using --no-share to avoid Gradio's own public URL, as we are using ngrok.
    # The --listen flag is crucial for allowing external connections via ngrok.
    subprocess.run([
        "python", "launch.py",
        "--listen",
        "--port", "7860",
        "--enable-insecure-extension-access",
        "--no-download-sd-model",
        "--xformers",
        "--skip-torch-cuda-test",
        "--clip-models-path", "/kaggle/working/clip-vit-large-patch14"
    ])

def wait_for_server(port, timeout=120):
    """Waits for the server to start listening on the specified port."""
    start_time = time.time()
    while True:
        if time.time() - start_time > timeout:
            raise TimeoutError("Server did not start within the specified timeout.")
        try:
            requests.get(f"http://localhost:{port}")
            print(f"Server is up and running on port {port}.")
            return True
        except requests.exceptions.ConnectionError:
            print("Waiting for WebUI server to start...")
            time.sleep(20)

# --- Main execution flow ---

# 1. Clean up existing tunnels to avoid conflicts.
print("Cleaning up existing ngrok tunnels...")
kill_existing_tunnels()

# 2. Start the WebUI in a background thread.
print("Starting Automatic1111 WebUI...")
webui_thread = threading.Thread(target=run_webui, daemon=True)
webui_thread.start()

# 3. Wait for the WebUI server to become available.
try:
    wait_for_server(7860)
except TimeoutError as e:
    print(f"Error: {e}")
    # You may want to stop the script here if the server fails to start.
    exit(1)

# 4. Create and display the ngrok tunnel.
print("Creating ngrok tunnel...")
try:
    public_url = ngrok.connect(7860)
    print(f"\n🎉 Stable Diffusion WebUI is accessible at:")
    print(f"🔗 {public_url}")
    print("\nClick the link above to access the WebUI interface!")

    # Keep the main thread alive so the WebUI thread continues to run
    # until the notebook cell execution is stopped.
    webui_thread.join()

except Exception as e:
    print(f"Error creating tunnel: {e}")
    print("Check your ngrok authentication token or try restarting the session.")